## 1. Problem Statement


Diabetes is a chronic disease affecting millions worldwide.This project builds an Artificial Neural Network (ANN) that predicts whether a patient is likely to have diabetes based on diagnostic measurements. This is a binary classification problem: Class 0 = No diabetes, Class 1 = Diabetes.


## 2. Dataset Description

Pima Indians Diabetes Dataset — 768 rows, 8 input features + 1 target (Outcome):
Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age, Outcome (1=diabetic, 0=non-diabetic).

## 3. Import libraries

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score, precision_score, recall_score, f1_score)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings("ignore")

print("TensorFlow Version:", tf.__version__)


## 4.Import Datasets

In [ ]:
df = pd.read_csv("Downloads/diabetes.csv")

df.head()

## 5. Data Analysis

In [ ]:
print(df.dtypes)
print()
print(df.describe())
print()
df.info()
print()
print("Rows and Columns")
print(df.shape)
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Class distribution (Outcome):")
print(df["Outcome"].value_counts())

## 6. EDA and Visualization

In [ ]:
# 1. Count plot of the target variable
sns.countplot(x="Outcome", data=df, palette="rainbow")
plt.title("Class Distribution: Diabetic (1) vs Non-Diabetic (0)")
plt.show()

In [ ]:
# 2. Boxplot for Glucose grouped by Outcome
sns.boxplot(x="Outcome", y="Glucose", data=df, palette="viridis")
plt.title("Glucose Levels vs Outcome")
plt.show()

In [ ]:
# 3. Age distribution by Outcome
sns.histplot(data=df, x="Age", hue="Outcome", kde=True, palette="inferno_r")
plt.title("Age Distribution by Outcome")
plt.show()

In [ ]:
# 4. Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
# 5. BMI distribution
sns.set(style="whitegrid")
ax = sns.displot(df["BMI"], kde=True, color="c")
plt.title("Distribution of BMI")
plt.show()

In [ ]:
# 6. Pairplot
sns.pairplot(df[["Glucose", "BMI", "Age", "Insulin", "Outcome"]], hue="Outcome", palette="husl")
plt.show()

## 7. Data Preprocessing

In [ ]:
cols_with_invalid_zeros = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

df[cols_with_invalid_zeros] = df[cols_with_invalid_zeros].replace(0, np.nan)

print("Missing values after marking invalid zeros as NaN:")
print(df[cols_with_invalid_zeros].isnull().sum())

for col in cols_with_invalid_zeros:
    df[col].fillna(df[col].median(), inplace=True)

print()
print("Missing values after imputation:")
print(df.isnull().sum())

## 8. Splitting Features and Target

In [ ]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

## 9. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training Features:", X_train.shape)
print("Training Labels  :", y_train.shape)
print("Testing Features :", X_test.shape)
print("Testing Labels   :", y_test.shape)

y_train.value_counts()

## 10. Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Sample scaled training data:")
print(X_train_scaled[:5])

## 11. ANN Model Building

In [ ]:
model = Sequential([
    Dense(16, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(8, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 12. Model Training

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

## 13. Training and Validation Graphs

In [ ]:
# Accuracy curve
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model Accuracy over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
# Loss curve
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Model Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 14. Accuracy / Loss Analysis

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")

## 15. Predictions on Test Set

In [ ]:
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print("Sample predicted probabilities:", y_pred_prob[:5].flatten())
print("Sample predicted classes      :", y_pred[:5])
print("Sample actual classes         :", y_test.values[:5])

## 16. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Diabetic", "Diabetic"],
            yticklabels=["Non-Diabetic", "Diabetic"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## 17. Precision, Recall, F1-Score

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")

print()
print("Full Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Non-Diabetic", "Diabetic"]))

## 18. Prediction on a New Sample

In [ ]:
# [Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age]
new_sample = np.array([[2, 150, 80, 30, 100, 33.6, 0.627, 45]])

new_sample_scaled = scaler.transform(new_sample)

prediction_prob = model.predict(new_sample_scaled)
prediction_class = (prediction_prob > 0.5).astype(int)

print(f"Predicted Probability of Diabetes: {prediction_prob[0][0]:.4f}")
if prediction_class[0][0] == 1:
    print("Prediction: The patient is likely DIABETIC.")
else:
    print("Prediction: The patient is likely NON-DIABETIC.")

## 19. Conclusion

An ANN was built with TensorFlow to predict diabetes on the Pima Indians dataset. Preprocessing (fixing invalid zero values) and StandardScaler-based feature scaling were essential since ANNs are sensitive to feature magnitude. Training/validation curves show the learning behavior and help detect overfitting. The confusion matrix and precision/recall/F1-score give a fuller picture of performance than accuracy alone, especially with imbalanced classes. The model was tested on a new sample to confirm real-world usability. Future work: hyperparameter tuning, deeper architectures, SMOTE for class imbalance, cross-validation.